# DESC 3x2pt Static-Probes FoM (GP emulator) versus the E(B-V) footprint threshold

``` 
OK, well here is what I’d like to modify.  In this code: https://github.com/lsst/rubin_sim/blob/8528c01602de0f9641cc1b58dce0b85dc07e945d/rubin_sim/maf/batches/science_radar_batch.py#L534
it sets configuration parameters that go into the area cuts in the metric.  In particular, I’m concerned that the numbers in lines 535-544 were set based on a much earlier strategy and they could be leading to area reductions unnecessarily.  One way to test this would be to set offset in line 533 to a large negative number, so that the cuts that use lines 535-544 aren’t removing any area.  If you could see what the value is of ExgalM5WithCuts before and after this change in the code, that would be very helpful — it would tell us how much area is being affected by strategy changes, versus by a potentially outdated cut inside of this metric.  Does that make sense and is it straightforward to do?
```

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-29
- Kernel: conda_py313_opsim53_scoctaskforce
- Context: SCOC footprint-shrinking study + DESC static-probes metrics. This notebook extends the `10_DESCMAFDEPTHANDGALCOUNT` series (E(B-V) footprint-threshold comparison of `ExgalM5WithCuts`, the galaxy-count metrics and the weak-lensing proxy metrics) to the **DESC 3x2pt static-probes Figure of Merit**, `StaticProbesFoMEmulatorMetric`.
- This notebook: **06** of the series - `StaticProbesFoMEmulatorMetric` (Gaussian-Process emulator, the summary metric used by the official `rubin_sim.maf.batches.science_radar_batch` "Cosmology" group) recomputed **for every year (1-10) and on each of the 7 dust-threshold footprint variants** of the v5.3.6 simulations.
- Companion notebooks: `01_compareExgalM5withCuts.ipynb` (the `ExgalM5WithCuts` parent metric, on the 7 runs at year 10 only), `03_WeakLensing.ipynb` (WL proxy metrics, same 7 runs). Reference notebook for the GP-emulator usage: `../06_MAF_DESC_TaskF/01b_3x2pts_DESC_TaskForce_demo.ipynb` (baseline only, year by year).
- Presentation model: `../07_variateEVmV/01_FOMNv_HealpixDiff_ShrinkFPDust.ipynb` (adapted here: the FoM is a single scalar per run and per year, not a Healpix map, so the "consecutive-difference" maps of notebooks 01-03 become curves and a heatmap versus (run, year) instead).
- OpSim simulations analyzed (`/Users/dagoret/DATA/OpSim/`):
  - `shrink_fp_dust_0.050_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.080_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.120_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.150_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.199_v5.3.6_10yrs.db`
  - `baseline_v5.3.6_10yrs.db` (E(B-V) threshold 0.200)
  - `shrink_fp_dust_0.250_v5.3.6_10yrs.db`


## Notebook overview

**What is combined here.** Two things that were so far studied separately in this package:

1. The **year-dependent** `ExgalM5WithCuts` -> `StaticProbesFoMEmulatorMetric` chain of `../06_MAF_DESC_TaskF/01b_3x2pts_DESC_TaskForce_demo.ipynb` (run on the baseline only, years 1 to 10).
2. The **E(B-V) footprint-threshold** comparison of notebooks 01 and 03 of this series (7 runs, year 10 only, `EBV_CUT_MODE = 'run'`: the metric's dust cut follows the WFD threshold that defines the footprint of each run).

For every one of the 7 runs and every one of the 10 years, this notebook builds the same `ExgalM5WithCuts` Healpix map as notebook 01 (same `n_filters`, `nside`, non-DD 10-year selection, per-year `depth_cut` taken from the official batch), and attaches as `summary_metrics`:
- `StaticProbesFoMEmulatorMetric(nside=nside, metric_name="3x2ptFoM_GP")` - the Gaussian-Process emulator, defined at every year (this notebook's subject);
- `StaticProbesFoMEmulatorMetricSimple(year=year, metric_name="3x2ptFoM_simple")` - the bilinear emulator used in `01_3x2pts_DESC_TaskForce_demo.ipynb`, only meaningful at years 1, 3, 6, 10, kept here as a cross-check;
- `MeanMetric`, `MedianMetric`, `RmsMetric` and a `CountRatioMetric` giving the effective area, exactly as in `01b`.

**Choices made in this notebook (all editable in Section 3), identical to notebooks 01 and 03.**
- The metric-side dust cut (`extinction_cut` of `ExgalM5WithCuts`) is **adapted to each run**: `EBV_CUT_MODE = 'run'` sets it to the E(B-V) threshold that defines the WFD footprint of that run (0.050, 0.080, 0.120, 0.150, 0.199, 0.200 for the baseline, 0.250). `EBV_CUT_MODE = 'fixed'` uses the same cut `FIXED_LIM_EBV = 0.2` for every run and every year, i.e. exactly the configuration of `01b_3x2pts_DESC_TaskForce_demo.ipynb` applied to all 7 runs.
- `bandpass = 'i'`, `n_filters = 6`, `nside = 64`, per-year `mag_cuts` and `offset = 0.1` copied verbatim from `01b`. All runs are truncated year by year with `night <= year*365.25 + 0.5` and `scheduler_note not like 'DD%'` (no separate 10-year truncation is needed here: the year-10 selection already is the "first 10 years" case, unlike notebooks 01/03 which fix `depth_cut` at the year-10 value and only run at year 10).

**How the figures are built.**
1. Results table (run x year): effective area, median depth, `3x2ptFoM_GP`, `3x2ptFoM_simple` where defined.
2. `3x2ptFoM_GP` versus year, one curve per run (color = E(B-V) threshold of the run), and the same curves normalized to the baseline FoM of the same year.
3. `3x2ptFoM_GP` versus the E(B-V) threshold, one curve per year (cross-sections of the same grid), and the normalized version.
4. A heatmap of `3x2ptFoM_GP` over the full (E(B-V) threshold, year) grid, and of its ratio to the baseline.
5. A consistency check against notebook 01 (same effective area and median depth at year 10, since both use `depth_cut = mag_cuts[10] = 25.9`).

**Caching.** MAF is run only once per (run, year); the resulting `summary_values` for all 10 years of a run are cached together as a single `.csv` in `data_06_FOM3X2PTS/` and reloaded at the next execution (set `FORCE_RECOMPUTE = True` to redo it). Within one run, the 10 yearly bundles are grouped into a single `MetricBundleGroup` so the OpSim database is read only once per run (7 reads total, not 70).

## Practical note before running

`StaticProbesFoMEmulatorMetric` needs `george` and `scikit-learn`, imported lazily inside its `run()` method, and refits its Gaussian-Process hyperparameters from scratch on every call. With 7 runs x 10 years = 70 calls, Section 6 below is the slow part of this notebook (typically a few minutes the first time; the cached `.csv` files make every later run instant). If `george`/`scikit-learn` are missing, install with `pip install george scikit-learn` (`george` needs a C/C++ compiler to build; on macOS with conda, `conda install -c conda-forge george` is usually the smoothest route).

In [ ]:
import importlib

for _pkg in ("george", "sklearn"):
    try:
        importlib.import_module(_pkg)
        print(f"{_pkg}: OK")
    except ImportError as e:
        print(
            f"{_pkg}: NOT FOUND ({e}). Install it before running Section 6 below "
            f"(e.g. 'pip install {_pkg}' or 'conda install -c conda-forge {_pkg}')."
        )

## 1. Imports

In [ ]:
import os
import time
import inspect
from os.path import join, isfile, splitext, basename

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metrics as metrics
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maf_maps
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.metrics.weak_lensing_systematics_metric import ExgalM5WithCuts
from rubin_sim.maf.metrics.cosmology_summary_metrics import StaticProbesFoMEmulatorMetricSimple
from rubin_sim.maf.maf_contrib.static_probes_fom_summary_metric import StaticProbesFoMEmulatorMetric

print("rubin_sim version:", rubin_sim.__version__)

## 2. Output directories

In [ ]:
NB_TAG = "FOM3X2PTS"
data_dir = f"data_06_{NB_TAG}"
figs_dir = f"figs_06_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. Configuration

In [ ]:
# Directory holding the OpSim databases
OPSIM_DIR = "/Users/dagoret/DATA/OpSim"

# (run_name, E(B-V) threshold that defines the footprint of the run), sorted by increasing threshold
# - identical list to notebooks 01 and 03 of this series
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_10yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
DUST_THRESH = dict(RUNS_INFO)
REF_RUN = "baseline_v5.3.6_10yrs"


def get_db_path(run_name):
    # Look in OPSIM_DIR, then in OPSIM_DIR/sim_baseline (where the baseline was stored in the 06/07 notebooks)
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run {run_name}. Tried: {candidates}")


for run_name, dust in RUNS_INFO:
    try:
        path = get_db_path(run_name)
    except FileNotFoundError:
        path = "NOT FOUND (only a problem if there is no cached result, see Section 6)"
    print(f"{run_name:40s} E(B-V) < {dust:.3f}  ->  {path}")

In [ ]:
BANDPASS = "i"
N_FILTERS = 6
NSIDE = 64  # same nside as science_radar_batch and as notebooks 01 and 03
OFFSET = 0.1  # same year-10 offset convention as 01b_3x2pts_DESC_TaskForce_demo.ipynb
# OFFSET = -0.5

# Per-year i-band coadded depth cut of the official science_radar_batch, minus OFFSET - copied from 01b
MAG_CUTS = {
    1: 24.75 - OFFSET,
    2: 25.12 - OFFSET,
    3: 25.35 - OFFSET,
    4: 25.50 - OFFSET,
    5: 25.62 - OFFSET,
    6: 25.72 - OFFSET,
    7: 25.80 - OFFSET,
    8: 25.87 - OFFSET,
    9: 25.94 - OFFSET,
    10: 26.00 - OFFSET,
}
YEARS = sorted(MAG_CUTS.keys())

# The Simple bilinear emulator is only calibrated at these years (cross-check only, as in 01b):
SIMPLE_FOM_DEFINED_YEARS = [1, 3, 6, 10]
# The GP emulator has no such restriction: it is evaluated at every year for every run.

# E(B-V) cut applied by the METRIC (extinction_cut of ExgalM5WithCuts), as in notebooks 01 and 03:
#   'run'   -> E(B-V) threshold that defines the WFD footprint of each run (DUST_THRESH), adapted run by run
#   'fixed' -> the same value FIXED_LIM_EBV for all runs (the configuration of 01b, extended to all 7 runs)
EBV_CUT_MODE = "fixed"
FIXED_LIM_EBV = 0.2

FORCE_RECOMPUTE = True  # True -> ignore the cached .csv results and rerun MAF

pix_area = hp.nside2pixarea(NSIDE, degrees=True)
print(f"nside={NSIDE} -> npix={hp.nside2npix(NSIDE)}, pixel area = {pix_area:.4f} deg^2")
print(f"Years evaluated: {YEARS}")
print(f"Simple emulator cross-check only at years: {SIMPLE_FOM_DEFINED_YEARS}")


def metric_ebv_cut(run_name):
    # E(B-V) cut given to the metric for this run
    if EBV_CUT_MODE == "run":
        return DUST_THRESH[run_name]
    if EBV_CUT_MODE == "fixed":
        return FIXED_LIM_EBV
    raise ValueError(f"EBV_CUT_MODE must be 'run' or 'fixed', got {EBV_CUT_MODE!r}")


def config_tag(run_name):
    # Cache tag; it contains the E(B-V) cut actually used, so a change of EBV_CUT_MODE never reloads
    # a result computed with another cut
    return f"{BANDPASS}_nf{N_FILTERS}_ebv{metric_ebv_cut(run_name):.3f}_nside{NSIDE}"


print(f"E(B-V) cut given to the metric (mode '{EBV_CUT_MODE}'):")
for run_name, _ in RUNS_INFO:
    print(
        f"  {run_name:40s} extinction_cut = {metric_ebv_cut(run_name):.3f}   cache tag: {config_tag(run_name)}"
    )

## 4. The metric classes used

`ExgalM5WithCuts` (parent, per-pixel) is unchanged from notebooks 01 and 03. The summary metric of interest is `StaticProbesFoMEmulatorMetric`; `StaticProbesFoMEmulatorMetricSimple` is kept only as a cross-check at years 1, 3, 6, 10. See `inspect.getdoc` / `inspect.getsource` in `00_DepthsAndCountsMetrics.ipynb` and `01_compareExgalM5withCuts.ipynb` (Section 3) for `ExgalM5WithCuts` itself; here we focus on the FoM emulator.

In [ ]:
print(inspect.getdoc(StaticProbesFoMEmulatorMetric))

In [ ]:
# The 36-point (area, depth, systematics) -> FoM training grid the GP interpolates over,
# and the default systematics values used when they are not explicitly overridden (same as 01b):
_probe = StaticProbesFoMEmulatorMetric()
grid_df = pd.DataFrame(_probe.parameters)
print("Training grid ranges:")
print(grid_df[["area", "depth", "FOM"]].describe().loc[["min", "max"]])
print()
print("Default systematics used by this notebook (same as the official batch and as 01b):")
print(f"  shear_m      = {_probe.shear_m}")
print(f"  sigma_z      = {_probe.sigma_z}")
print(f"  sig_delta_z  = {_probe.sig_delta_z}")
print(f"  sig_sigma_z  = {_probe.sig_sigma_z}")

## 5. Helper functions

Same plotting/saving conventions as notebooks 01 and 03. Since the FoM is a scalar (one value per run and per year, not a Healpix map), no mask/diff-map helpers are needed here; the closest thing to notebooks 01-03's "consecutive difference" maps is the (E(B-V) threshold, year) heatmap of Section 9.

In [ ]:
def run_label(run_name):
    label = f"E(B-V) < {DUST_THRESH[run_name]:.3f}"
    return label + " (baseline)" if run_name.startswith("baseline") else label


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


dustmap = maf_maps.DustMap(nside=NSIDE, interp=False)


def build_bundle(run_name, year, ebv_cut):
    # Same ExgalM5WithCuts chain as notebook 01 (year-10 depth cut only) and 01b (all years),
    # here combined: per-year depth cut (MAG_CUTS) x per-run E(B-V) cut (ebv_cut).
    depth_cut = MAG_CUTS[year]
    sqlconstraint = "night <= %s and scheduler_note not like 'DD%%'" % (year * 365.25 + 0.5)
    info_label = f"{BANDPASS} band non-DD year {year} ebv{ebv_cut:.3f}"

    parent_metric = metrics.ExgalM5WithCuts(
        lsst_filter=BANDPASS,
        n_filters=N_FILTERS,
        extinction_cut=ebv_cut,
        depth_cut=depth_cut,
    )
    summary_metrics = [
        metrics.MeanMetric(),
        metrics.MedianMetric(),
        metrics.RmsMetric(),
        metrics.CountRatioMetric(norm_val=1.0 / pix_area, metric_name="Effective Area (deg)"),
        StaticProbesFoMEmulatorMetric(nside=NSIDE, metric_name="3x2ptFoM_GP"),
        StaticProbesFoMEmulatorMetricSimple(year=year, metric_name="3x2ptFoM_simple"),
    ]
    slicer = slicers.HealpixSlicer(nside=NSIDE, use_cache=False)
    return mb.MetricBundle(
        parent_metric,
        slicer,
        sqlconstraint,
        maps_list=[dustmap],
        run_name=run_name,
        info_label=info_label,
        summary_metrics=summary_metrics,
    )


def cache_path(run_name):
    tag = run_name.replace(".", "_")
    return join(data_dir, f"FOM3x2pt_{tag}_{config_tag(run_name)}.csv")


def compute_run(run_name):
    # All 10 years of one run are grouped into a single MetricBundleGroup, so the OpSim database
    # is opened only once per run (7 opens total for the whole notebook, not 70).
    ebv_cut = metric_ebv_cut(run_name)
    dbpath = get_db_path(run_name)
    print(
        f"[{run_name}] running ExgalM5WithCuts + FoM emulators for {len(YEARS)} years (extinction_cut = {ebv_cut:.3f}) on {dbpath}"
    )
    bundles = {year: build_bundle(run_name, year, ebv_cut) for year in YEARS}
    group = mb.MetricBundleGroup(
        mb.make_bundles_dict_from_list(list(bundles.values())), dbpath, out_dir=data_dir, results_db=resultsDb
    )
    group.run_all()

    rows = []
    for year in YEARS:
        sv = bundles[year].summary_values
        rows.append(
            {
                "year": year,
                "i_depth_cut": MAG_CUTS[year],
                "effective_area_deg2": sv.get("Effective Area (deg)"),
                "median_depth": sv.get("Median"),
                "rms_depth": sv.get("Rms"),
                "fom_3x2pt_GP": sv.get("3x2ptFoM_GP"),
                "fom_3x2pt_simple": sv.get("3x2ptFoM_simple") if year in SIMPLE_FOM_DEFINED_YEARS else np.nan,
            }
        )
    return pd.DataFrame(rows).set_index("year")


def load_or_compute(run_name):
    path = cache_path(run_name)
    if isfile(path) and not FORCE_RECOMPUTE:
        print(f"[{run_name}] loaded cached results: {path}")
        return pd.read_csv(path, index_col="year")
    df = compute_run(run_name)
    df.to_csv(path)
    print(f"[{run_name}] saved: {path}")
    return df

## 6. Compute (or reload) the FoM grid: 7 runs x 10 years

First execution: 7 x 10 = 70 `ExgalM5WithCuts` + GP-emulator evaluations (a few minutes, dominated by the GP hyperparameter fits). Cached afterwards as one `.csv` per run in `data_06_FOM3X2PTS/`.

In [ ]:
t0 = time.time()
results_by_run = {run_name: load_or_compute(run_name) for run_name, _ in RUNS_INFO}
print(f"Done in {time.time() - t0:.1f} s")

In [ ]:
combined = pd.concat(results_by_run, names=["run", "year"])
combined["ebv_threshold"] = combined.index.get_level_values("run").map(DUST_THRESH)
combined_csv = join(data_dir, "FOM3x2pt_GP_all_runs_all_years.csv")
combined.to_csv(combined_csv)
print("Saved:", combined_csv)
combined.round(3)

## 7. `3x2ptFoM_GP` versus survey year, one curve per run

Color = E(B-V) threshold of the run (same `viridis` convention as the overlay histograms of notebooks 01-03). Markers: `StaticProbesFoMEmulatorMetricSimple`, shown for the baseline run only (years 1, 3, 6, 10) as a cross-check against `../06_MAF_DESC_TaskF/01b_3x2pts_DESC_TaskForce_demo.ipynb`; it should reproduce that notebook's Section 8 exactly, since the baseline row uses the same `extinction_cut = 0.2`.

In [ ]:
colors = plt.cm.viridis(np.linspace(0.0, 0.95, len(RUNS_INFO)))

fig, ax = plt.subplots(figsize=(8, 6))
for (run_name, dust), c in zip(RUNS_INFO, colors):
    df = results_by_run[run_name]
    ax.plot(df.index, df["fom_3x2pt_GP"], marker="o", color=c, label=run_label(run_name))

baseline_df = results_by_run[REF_RUN]
simple_defined = baseline_df.loc[baseline_df.index.isin(SIMPLE_FOM_DEFINED_YEARS)]
# ax.plot(
#    simple_defined.index,
#    simple_defined["fom_3x2pt_simple"],
#    marker="s",
#    linestyle="none",
#    color="crimson",
#    markersize=9,
#    markerfacecolor="none",
#    label="StaticProbesFoMEmulatorMetricSimple (baseline, cross-check)",
# )
ax.set_xlabel("Survey year")
ax.set_ylabel("3x2pt static-probes FoM (GP emulator)")
ax.set_title("StaticProbesFoMEmulatorMetric versus year, for the 7 E(B-V) footprint variants")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
save_fig(fig, "FOM3x2pt_GP_vs_year_allruns")
plt.show()

### 7.1 Same plot, normalized to the baseline FoM of the same year

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ref = baseline_df["fom_3x2pt_GP"]
for (run_name, dust), c in zip(RUNS_INFO, colors):
    df = results_by_run[run_name]
    ax.plot(df.index, df["fom_3x2pt_GP"] / ref, marker="o", color=c, label=run_label(run_name))
ax.axhline(1.0, color="gray", ls="--", linewidth=0.8)
ax.set_xlabel("Survey year")
ax.set_ylabel("3x2pt FoM (GP) / baseline FoM, same year")
ax.set_title("StaticProbesFoMEmulatorMetric versus year, normalized to the baseline")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
save_fig(fig, "FOM3x2pt_GP_vs_year_allruns_norm")
plt.show()

## 8. `3x2ptFoM_GP` versus the E(B-V) footprint threshold, one curve per year

Cross-sections of the same (run, year) grid, this time with the E(B-V) threshold on the x-axis - the natural way to compare with the "metric vs E(B-V) threshold" plots of Sections 10-13 of notebooks 01 and 03. A subset of years is shown for readability; edit `YEARS_SHOWN` to change it.

In [ ]:
YEARS_SHOWN = [1, 3, 6, 10]
x = np.array([DUST_THRESH[r] for r in RUN_NAMES])
year_colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(YEARS_SHOWN)))

fig, ax = plt.subplots(figsize=(7, 5.5))
for year, c in zip(YEARS_SHOWN, year_colors):
    y = [results_by_run[r].loc[year, "fom_3x2pt_GP"] for r in RUN_NAMES]
    ax.plot(x, y, marker="o", color=c, label=f"year {year}")
i_ref = RUN_NAMES.index(REF_RUN)
ax.axvline(x[i_ref], color="gray", ls="--", linewidth=0.8, label="baseline threshold")
ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
ax.set_ylabel("3x2pt static-probes FoM (GP emulator)")
ax.set_title("StaticProbesFoMEmulatorMetric versus E(B-V) footprint threshold")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, "FOM3x2pt_GP_vs_dust_threshold_byyear")
plt.show()

### 8.1 Same plot, normalized to the baseline run (same year)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
for year, c in zip(YEARS_SHOWN, year_colors):
    y = np.array([results_by_run[r].loc[year, "fom_3x2pt_GP"] for r in RUN_NAMES])
    ax.plot(x, y / y[i_ref], marker="o", color=c, label=f"year {year}")
ax.axhline(1.0, color="gray", ls="--", linewidth=0.8)
ax.axvline(x[i_ref], color="gray", ls="--", linewidth=0.8, label="baseline")
ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
ax.set_ylabel("3x2pt FoM (GP) / baseline FoM, same year")
ax.set_title("StaticProbesFoMEmulatorMetric versus E(B-V) footprint threshold, normalized")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, "FOM3x2pt_GP_vs_dust_threshold_byyear_norm")
plt.show()

## 9. Heatmap of `3x2ptFoM_GP` over the full (E(B-V) threshold, year) grid

Left: absolute FoM. Right: ratio to the baseline row (year by year), the 2-D equivalent of Sections 7.1/8.1. Rows are sorted by increasing E(B-V) threshold, columns by year.

In [ ]:
grid = np.array([[results_by_run[r].loc[y, "fom_3x2pt_GP"] for y in YEARS] for r in RUN_NAMES])
grid_norm = grid / grid[i_ref : i_ref + 1, :]

fig, axs = plt.subplots(1, 2, figsize=(13, 5))

im0 = axs[0].imshow(grid, aspect="auto", cmap="viridis", origin="lower")
axs[0].set_title("3x2ptFoM_GP")
fig.colorbar(im0, ax=axs[0], label="FoM")

im1 = axs[1].imshow(
    grid_norm, aspect="auto", cmap="RdBu_r", origin="lower", vmin=2 - grid_norm.max(), vmax=grid_norm.max()
)
axs[1].set_title("3x2ptFoM_GP / baseline (same year)")
fig.colorbar(im1, ax=axs[1], label="ratio")

for ax in axs:
    ax.set_xticks(range(len(YEARS)))
    ax.set_xticklabels(YEARS)
    ax.set_xlabel("Survey year")
    ax.set_yticks(range(len(RUN_NAMES)))
    ax.set_yticklabels([f"{DUST_THRESH[r]:.3f}" for r in RUN_NAMES])
    ax.set_ylabel("E(B-V) threshold")
    ax.axhline(i_ref, color="k", linewidth=0.8, linestyle="--")

fig.suptitle("StaticProbesFoMEmulatorMetric over the full (E(B-V) threshold, year) grid", y=1.03)
fig.tight_layout()
save_fig(fig, "FOM3x2pt_GP_heatmap_dustthreshold_vs_year")
plt.show()

## 10. Consistency check against notebook 01

At year 10, `depth_cut = MAG_CUTS[10] = 25.9`, the same value as `DEPTH_CUT` in `01_compareExgalM5withCuts.ipynb`, and `EBV_CUT_MODE = 'run'` gives the same `extinction_cut` per run. The `effective_area_deg2` and `median_depth` columns of this notebook's year-10 row should therefore match notebook 01's `per_run_summary` table (`area_deg2` and `median`) to within the small numerical differences expected between independent `HealpixSlicer` runs (scheduler/MAF non-determinism is not expected here since both notebooks read the same OpSim databases; residual differences, if any, would point to a mismatch in one of the two configurations).

In [ ]:
year10 = combined.xs(10, level="year")[
    ["ebv_threshold", "effective_area_deg2", "median_depth", "fom_3x2pt_GP", "fom_3x2pt_simple"]
]
year10 = year10.reindex(RUN_NAMES)
year10.round(3)

## Caveats

- The GP is trained on a 36-point grid spanning `area` in roughly [7600, 19800] deg^2 and `depth` in roughly [24.8, 26.8] mag (Section 4). Early-year and/or small-footprint (low E(B-V) threshold) cells of the grid can fall near or outside the edge of this training range, so the GP prediction there is an **extrapolation** and should be read with caution.
- The GP model itself has **no notion of survey year or E(B-V) threshold** - it only ever sees the (area, depth) point computed from the map that was fed to it. Two different (run, year) cells that happen to produce the same (area, depth) get the same FoM; all of the structure seen in Sections 7-9 comes from how (area, depth) evolve with the year and the dust threshold, not from anything built into the emulator.
- Systematics (`shear_m`, `sigma_z`, `sig_delta_z`, `sig_sigma_z`) are left at their defaults, matching the official batch and `01b`.
- With `EBV_CUT_MODE = 'run'` the metric cut follows the threshold of each run, so the metric footprint approximately reproduces the WFD footprint scheduled in that run; it is not guaranteed to be identical to it (see the caveats of notebook 01). With `EBV_CUT_MODE = 'fixed'` every run is evaluated with `extinction_cut = 0.2`, i.e. this notebook reduces to `01b` applied 7 times to different visit databases with the same footprint definition.
- `0.199` and `baseline (0.200)` have almost the same nominal footprint: the difference between their FoM curves gives the scale of the run-to-run (scheduler) fluctuations at fixed E(B-V) threshold, to compare against the change seen between the other, more widely separated thresholds.
- The GP hyperparameters are refit from scratch on every single call, so this notebook makes 70 independent fits; this is the dominant cost of Section 6, though still far cheaper than `SNNSNMetric` (see `../06_MAF_DESC_TaskF/03_SN_DESC_TaskForce_demo.ipynb`).

## References
- Lochner, M. et al. 2018, "Optimizing LSST Observing Strategy for Dark Energy Science", arXiv:1808.00006 - defines the static-probes (3x2pt) Figure-of-Merit forecast grid used by both emulators.
- Zuntz, J. et al. 2021, "The LSST-DESC 3x2pt Tomography Optimization Challenge", arXiv:2108.13418.
- Bianco, F. B. et al. 2022, ApJS 258, 1 - SCOC context.
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/maf_contrib/static_probes_fom_summary_metric.py`, `rubin_sim/maf/metrics/cosmology_summary_metrics.py`, `rubin_sim/maf/metrics/weak_lensing_systematics_metric.py`, `rubin_sim/maf/batches/science_radar_batch.py`)
- `../06_MAF_DESC_TaskF/01b_3x2pts_DESC_TaskForce_demo.ipynb` - the GP-emulator chain, baseline only, year by year (this notebook's direct model).
- `01_compareExgalM5withCuts.ipynb`, `03_WeakLensing.ipynb` - the E(B-V) footprint-threshold comparison, year 10 only (this notebook's other model).
- `shrink_fp_dust_*.db` simulations: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/shrink_fp/